# BBBP general local-BFS — fixed-Q, variable-T scan

This notebook holds Q fixed at `8, 12, 16, 20` and scans `T=1,...,30`. Within each Q panel, token width and reservoir/readout width are fixed; only the exposed task-independent BFS event prefix changes.

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'core' / 'linear_readout.py').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Could not locate 2_linear_readout project root')
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from core.linear_readout import (
    DEFAULT_SEEDS, ExperimentConfig, prepare_fixed_q_layout,
    run_scan, collect_scan_results, plot_fixed_q_scan,
)
print('project:', PROJECT_ROOT)
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())

## Fixed protocol

QRC and old scalar-state CRC are aligned in input preprocessing, Q, layers, and final feature dimension. No three-axis CRC is used. The direct raw-full/raw-PCA baselines flattens the original token tensor without reservoir preprocessing before the same one-layer classifier.

In [ ]:
SCAN_ROOT = PROJECT_ROOT / 'general' / 'BBBP' / '02_fixed_Q_T_scan'
DATA_DIR = PROJECT_ROOT / 'general' / 'BBBP' / 'data'
SETTINGS = prepare_fixed_q_layout(SCAN_ROOT)
CFG = ExperimentConfig(
    dataset='BBBP', data_dir=str(DATA_DIR), scan_root=str(SCAN_ROOT),
    epochs=500, seeds=DEFAULT_SEEDS, W=0.4, g=0.84, layers=3,
    qrc_devices=('cuda:0', 'cuda:1'), run_qrc=True, force=True,
)
print(CFG)
print('settings:', len(SETTINGS), SETTINGS[0], '...', SETTINGS[-1])

## Train and evaluate

Every Q/T pair has a separate folder and complete per-seed audit trail. Rerunning the cell skips valid completed artifacts and continues from the first missing seed or setting.

In [ ]:
results = run_scan(CFG, SETTINGS)
results.tail(12)

## Result visualization

Rows show ROC-AUC and accuracy; columns fix Q. Curves are ten-seed means and bands are one standard deviation.

In [ ]:
summary = collect_scan_results(SCAN_ROOT)
display(summary)
fig = plot_fixed_q_scan(SCAN_ROOT, 'BBBP')
fig